# Boruta + Random Forest gene panel (secondary notebook)

Reads the output of **`pd-lcm-rf-core`**: the same 63 people, the same folds, and the
core model's gene importance.

1. **Honest panel performance** - Boruta is re-run from scratch inside every training fold
   (standard setting perc 100, and the broadened perc 99 used for the reported panel); a
   Random Forest on the confirmed genes scores the held-out people.
2. **The reported panel** - Boruta on all 63 people, a Random Forest on the confirmed
   genes, explained with TreeSHAP; how often each gene was also confirmed inside the folds.
3. **Agreement** with the core model's top genes, and single-gene ROCs.
4. **Tables** for Figures 4 and 5 and the performance table.

In [ ]:
import subprocess, sys, os, time, json, glob, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
for _a, _t in [("float", float), ("int", int), ("bool", bool), ("object", object), ("str", str)]:
    if not hasattr(np, _a):
        setattr(np, _a, _t)
ON_KAGGLE = Path("/kaggle/input").exists()
SMOKE = not ON_KAGGLE
OUT = Path("/kaggle/working") if ON_KAGGLE else Path(os.environ.get("SMOKE_OUT", "smoke_out"))
OUT.mkdir(exist_ok=True, parents=True)
LOCAL_IN = os.environ.get("SMOKE_IN", "data").split(":")
N_CPU = os.cpu_count()
SEED = 42
t0 = time.time()
def log(m): print(f"[{time.time() - t0:6.0f}s] {m}", flush=True)

def find_input(pattern):
    roots = ("/kaggle/input",) if ON_KAGGLE else tuple(LOCAL_IN)
    for root in roots:
        hits = sorted(glob.glob(f"{root}/**/{pattern}", recursive=True), key=len)
        if hits:
            return hits[0]
    raise FileNotFoundError(f"{pattern} not found under {roots}")
print("Kaggle" if ON_KAGGLE else "LOCAL SMOKE RUN", "| CPUs:", N_CPU)
try:
    import boruta
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "Boruta==0.4.3"], check=True)
from boruta import BorutaPy
import shap
from scipy.stats import hypergeom
from sklearn.model_selection import StratifiedKFold
N_TREES   = 40 if SMOKE else 1000   # trees in the panel forest
B_TREES   = 40 if SMOKE else 500    # trees per Boruta iteration
B_ITER    = 25 if SMOKE else 100     # Boruta iterations
MAIN_PERC = 99                      # reported panel; perc 100 (strictest) is scored alongside

In [ ]:
# ============================== LOAD THE CORE NOTEBOOK'S OUTPUT ==============================
cz = np.load(find_input("core_data.npz"), allow_pickle=True)
X, XR, y = cz["X"], cz["XR"], cz["y"].astype(int)
DS, PERSON, GENES = cz["ds"].astype(str), cz["person"].astype(str), [str(g) for g in cz["genes"]]
STRATA = np.array([f"{d}_{v}" for d, v in zip(DS, y)])
FOLDS = json.load(open(find_input("core_folds.json")))
for f in FOLDS:
    f["train"], f["test"] = np.array(f["train"]), np.array(f["test"])
SYM = pd.read_csv(find_input("15_gene_symbol_map.csv")).set_index("gene")["symbol"].to_dict()
sym = lambda g: SYM[g] if isinstance(SYM.get(g), str) else g
print(f"core data: {len(y)} people, {X.shape[1]:,} genes, {sum(f['kind'] == 'cv' for f in FOLDS)} CV folds + "
      f"{sum(f['kind'] == 'lodo' for f in FOLDS)} leave-one-study-out folds")

In [ ]:
from scipy.stats import rankdata
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import (roc_auc_score, roc_curve, accuracy_score, balanced_accuracy_score, recall_score,
                             f1_score, matthews_corrcoef, brier_score_loss)

HEAD_NAME = "Ranks + PCA 30 + Random Forest"
HEAD = ("rpca", 30, dict(max_features=0.5, min_samples_leaf=1))

def rf(n_jobs=1, **kw):
    p = dict(n_estimators=N_TREES, max_features="sqrt", min_samples_leaf=1, class_weight="balanced",
             oob_score=True, n_jobs=n_jobs, random_state=SEED)
    p.update(kw); return RandomForestClassifier(**p)

def features(kind, arg, tr, te):
    if kind == "all":
        return X[tr], X[te]
    if kind == "genes":
        return X[tr][:, arg], X[te][:, arg]
    src = XR if kind == "rpca" else X
    p = PCA(arg, random_state=SEED).fit(src[tr])
    return p.transform(src[tr]), p.transform(src[te])

def oob_threshold(m, yt):
    """Accuracy cut-off chosen on the forest's out-of-bag predictions for the training people."""
    s = m.oob_decision_function_[:, 1]; ok = np.isfinite(s); s, yt = s[ok], yt[ok]
    u = np.unique(np.round(s, 6))
    cuts = np.concatenate([[-np.inf], (u[:-1] + u[1:]) / 2, [np.inf]]) if len(u) > 1 else np.array([0.5])
    acc = [accuracy_score(yt, (s > c).astype(int)) for c in cuts]
    best = np.flatnonzero(np.isclose(acc, max(acc)))
    return float(cuts[best[len(best) // 2]])

def fit_score(spec, tr, te, yy, n_jobs=1):
    kind, arg, kw = spec
    Xtr, Xte = features(kind, arg, tr, te)
    m = rf(n_jobs=n_jobs, **kw).fit(Xtr, yy[tr])
    return m.predict_proba(Xte)[:, 1], oob_threshold(m, yy[tr])

def metrics(yt, s, th=None):
    yh = (s >= 0.5).astype(int)
    out = {"auc": roc_auc_score(yt, s), "accuracy": accuracy_score(yt, yh), "bal_accuracy": balanced_accuracy_score(yt, yh),
           "sensitivity": recall_score(yt, yh), "specificity": recall_score(1 - yt, 1 - yh),
           "f1": f1_score(yt, yh, zero_division=0), "mcc": matthews_corrcoef(yt, yh), "brier": brier_score_loss(yt, s)}
    if th is not None:
        out["accuracy_oob_cut"] = accuracy_score(yt, (s > th).astype(int))
    return out

def summarise(name, recs, key):
    """Metrics of one model over the fold records: mean over CV repeats, pooled over LODO folds."""
    reps = sorted({o["rep"] for o in recs if o["kind"] == "cv"})
    per = []
    for r in reps:
        fr = [o for o in recs if o["kind"] == "cv" and o["rep"] == r]
        idx = np.concatenate([o["test"] for o in fr]); s = np.concatenate([o[key]["score"] for o in fr])
        th = np.concatenate([np.full(len(o["test"]), o[key]["thr"]) for o in fr])
        per.append(metrics(y[idx], s, th))
    d = pd.DataFrame(per)
    lo = [o for o in recs if o["kind"] == "lodo"]
    li = np.concatenate([o["test"] for o in lo]); ls = np.concatenate([o[key]["score"] for o in lo])
    row = {"model": name, **{f"cv_{k}": d[k].mean() for k in d.columns},
           "cv_auc_sd": d.auc.std(ddof=1) if len(d) > 1 else np.nan,
           "cv_accuracy_sd": d.accuracy.std(ddof=1) if len(d) > 1 else np.nan,
           "lodo_auc": roc_auc_score(y[li], ls), "lodo_accuracy": accuracy_score(y[li], (ls >= 0.5).astype(int))}
    for o in lo:
        row[f"lodo_auc_{o['tag'][5:]}"] = roc_auc_score(y[np.array(o["test"])], o[key]["score"])
    return row

def boruta(idx, perc, seed=SEED):
    b = BorutaPy(RandomForestClassifier(max_features="sqrt", class_weight="balanced", n_jobs=N_CPU),
                 n_estimators=B_TREES, perc=perc, alpha=0.05, two_step=True, max_iter=B_ITER,
                 random_state=seed, verbose=0).fit(X[idx], y[idx])
    conf = np.where(b.support_)[0]
    used = conf if len(conf) >= 2 else np.where(b.support_ | b.support_weak_)[0]
    if len(used) < 2:
        used = np.argsort(b.ranking_)[:2]
    return b, conf, used

def tree_shap(model, Z):
    v = shap.TreeExplainer(model).shap_values(Z)
    v = v[1] if isinstance(v, list) else v
    return v[..., 1] if v.ndim == 3 else v

## 1. Boruta inside every fold

In [ ]:
PERCS = sorted({MAIN_PERC, 100})
REC = []
for i, f in enumerate(FOLDS):
    rec = {k: v for k, v in f.items() if k not in ("train", "test")}; rec["test"] = f["test"]
    for perc in PERCS:
        _, conf, used = boruta(f["train"], perc, f["seed"])
        s, t = fit_score(("genes", used, {}), f["train"], f["test"], y, n_jobs=N_CPU)
        rec[f"perc{perc}"] = {"score": s.tolist(), "thr": t, "genes": [GENES[j] for j in conf]}
    REC.append(rec)
    log(f"fold {i + 1}/{len(FOLDS)} {f['tag']}: " + ", ".join(f"{len(rec[f'perc{p}']['genes'])} genes (perc {p})" for p in PERCS))
json.dump([{**r, "test": r["test"].tolist()} for r in REC], open(OUT / "panel_fold_records.json", "w"))

rows = [summarise(f"Boruta (perc {p}) + Random Forest", REC, f"perc{p}") for p in PERCS]
CORE = pd.read_csv(find_input("core_performance.csv"))
PERF = pd.concat([CORE, pd.DataFrame(rows)], ignore_index=True).sort_values("cv_auc", ascending=False).reset_index(drop=True)
PERF.insert(0, "rank", range(1, len(PERF) + 1))
PERF.to_csv(OUT / "performance_all_models.csv", index=False)
pd.set_option("display.width", 220)
print(PERF[["rank", "model", "cv_auc", "cv_auc_sd", "cv_accuracy", "cv_bal_accuracy", "lodo_auc"]].round(3).to_string(index=False))

## 2. The reported panel on all 63 people

In [ ]:
ALL = np.arange(len(y))
_, conf100, _ = boruta(ALL, 100)
bor, conf, used_all = boruta(ALL, MAIN_PERC)
tent = np.where(bor.support_weak_)[0]
if len(conf) == 0:                                   # never expected on the full run; keeps smoke runs going
    print("no gene confirmed - using confirmed + tentative / best-ranked instead"); conf = used_all
panel = [GENES[j] for j in conf]
log(f"Boruta on all 63: {len(conf)} confirmed at perc {MAIN_PERC} ({len(tent)} tentative); {len(conf100)} at perc 100")
print(", ".join(sym(g) for g in panel))

panel_rf = rf(n_jobs=N_CPU).fit(X[:, conf], y)
SHAP = tree_shap(panel_rf, X[:, conf])
mean_abs = np.abs(SHAP).mean(0)
rho = np.array([pd.Series(X[:, j]).corr(pd.Series(SHAP[:, k]), method="spearman") for k, j in enumerate(conf)])
cv = [r for r in REC if r["kind"] == "cv"]
freq = pd.Series(0.0, index=GENES)
for r in cv:
    freq[r[f"perc{MAIN_PERC}"]["genes"]] += 1
freq /= len(cv)

## 3. Agreement with the core model, and single genes

In [ ]:
CG = pd.read_csv(find_input("core_gene_importance.csv")).set_index("gene")
N = len(panel); top_core = CG.sort_values("rank").index[:N].tolist()
ovl = sorted(set(panel) & set(top_core))
p_hyper = float(hypergeom.sf(len(ovl) - 1, len(GENES), N, N)) if N else 1.0
AGREE = {"panel_size": N, "overlap_with_core_topN": len(ovl), "expected_by_chance": N * N / len(GENES),
         "hypergeometric_p": p_hyper, "overlap_genes": [sym(g) for g in ovl],
         "panel_median_core_rank": float(CG.loc[panel, "rank"].median()) if N else None,
         "panel_core_ranks": {sym(g): int(CG.loc[g, "rank"]) for g in panel}}
json.dump(AGREE, open(OUT / "panel_core_agreement.json", "w"), indent=1)
log(f"{len(ovl)} of {N} panel genes are in the core model's top {N} (chance {AGREE['expected_by_chance']:.2f}, "
    f"p = {p_hyper:.2g}); median core rank of panel genes {AGREE['panel_median_core_rank']}")

# each gene's own expression as the score; its direction is learned on the training fold
def single_gene_oof(j, reps=10):
    P = np.zeros(len(y))
    for r in range(reps):
        for tr, te in StratifiedKFold(5, shuffle=True, random_state=SEED + r).split(X, STRATA):
            sgn = np.sign(X[tr][y[tr] == 1, j].mean() - X[tr][y[tr] == 0, j].mean()) or 1.0
            P[te] += sgn * X[te, j]
    return P / reps
rng = np.random.default_rng(SEED)
BOOT = [rng.integers(0, len(y), len(y)) for _ in range(200 if SMOKE else 2000)]
def auc_ci(p):
    a = roc_auc_score(y, p); flipped = a < 0.5
    if flipped:
        p = -p; a = 1 - a
    bs = [roc_auc_score(y[i], p[i]) for i in BOOT if len(set(y[i])) == 2]
    return a, np.percentile(bs, 2.5), np.percentile(bs, 97.5), flipped, p
sg_rows, roc_rows = [], []
for j in conf:
    a, lo, hi, fl, p = auc_ci(single_gene_oof(j))
    g = GENES[j]
    sg_rows.append({"gene": g, "symbol": sym(g), "auc": a, "ci_lo": lo, "ci_hi": hi, "direction_flipped": fl})
    fpr, tpr, _ = roc_curve(y, p)
    roc_rows += [{"curve": g, "curve_type": "single_gene", "fpr": a_, "tpr": b_, "auc": a, "ci_lo": lo, "ci_hi": hi}
                 for a_, b_ in zip(fpr, tpr)]
SG = pd.DataFrame(sg_rows).sort_values("auc", ascending=False).reset_index(drop=True)
print(SG[["symbol", "auc", "ci_lo", "ci_hi"]].round(3).to_string(index=False))

## 4. Tables for the figures

In [ ]:
NOMINAL_P = 0.01
de = pd.read_csv(find_input("03_de_results_full.csv")); de["is_deg"] = de["pvalue"] < NOMINAL_P
deg_set = set(de.loc[de.is_deg, "gene"]); bor_set = set(panel); overlap = bor_set & deg_set
imp = pd.DataFrame({"gene": panel, "symbol": [sym(g) for g in panel], "mean_abs_shap": mean_abs, "mean_shap": SHAP.mean(0),
                    "shap_std": SHAP.std(0), "shap_expression_spearman": rho,
                    "boruta_fold_frequency": freq[panel].to_numpy(), "core_model_rank": CG.loc[panel, "rank"].to_numpy()})
imp["direction_sign"] = np.sign(imp.shap_expression_spearman).astype(int)
imp["signed_importance"] = imp.direction_sign * imp.mean_abs_shap
imp["direction"] = np.where(imp.direction_sign > 0, "Higher expression raises PD probability",
                            "Higher expression lowers PD probability")
imp = imp.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
imp["shap_rank"] = range(1, len(imp) + 1)
deL = de.set_index("gene")
ann = imp.merge(SG[["gene", "auc", "ci_lo", "ci_hi"]].rename(columns={"auc": "single_gene_auc", "ci_lo": "single_gene_ci_lo",
                                                                     "ci_hi": "single_gene_ci_hi"}), on="gene")
for c in ("log2FC", "pvalue", "padj", "hedges_g_meta", "I2", "same_direction_datasets"):
    ann[c] = ann.gene.map(deL[c])
ann["is_deg"] = ann.gene.isin(deg_set); ann["in_overlap_panel"] = ann.gene.isin(overlap)
ann["in_core_topN"] = ann.gene.isin(top_core)
members = sorted(bor_set | deg_set)
memb = pd.DataFrame({"gene": members, "symbol": [sym(g) for g in members]})
memb["in_boruta"] = memb.gene.isin(bor_set); memb["in_deg"] = memb.gene.isin(deg_set); memb["in_overlap"] = memb.gene.isin(overlap)
for c in ("log2FC", "pvalue", "padj"):
    memb[c] = memb.gene.map(deL[c])

W = {}
def save(df, name, desc):
    df.to_csv(OUT / name, index=False); W[name] = (len(df), desc)
save(de, "03_de_results_full.csv", "DE across 63 people (merged pipeline; nominal p<0.01 flagged)")
save(pd.DataFrame([("de_rule", "nominal"), ("de_label", f"nominal DE genes (p < {NOMINAL_P}, not FDR-corrected)"),
                   ("nominal_p", NOMINAL_P), ("n_de_set", len(deg_set))], columns=["statistic", "value"]),
     "03_de_settings.csv", "DE rule used")
save(pd.DataFrame({"gene": panel, "symbol": [sym(g) for g in panel], "boruta_rank": bor.ranking_[conf], "confirmed": True}),
     "04_boruta_selected_genes.csv", f"Boruta + Random Forest (perc {MAIN_PERC}), all 63 people")
save(pd.DataFrame([("importance_model", f"Random Forest, {B_TREES} trees, max_features=sqrt, balanced"),
                   ("boruta_perc", MAIN_PERC), ("n_confirmed", len(conf)), ("n_tentative", len(tent)),
                   ("n_confirmed_perc100", len(conf100)),
                   ("mean_genes_per_training_fold", float(np.mean([len(r[f"perc{MAIN_PERC}"]["genes"]) for r in cv]))),
                   ("n_folds", len(cv))], columns=["setting", "value"]), "04_boruta_settings.csv", "Boruta settings and stability")
save(imp, "05_boruta_shap_importance.csv", "TreeSHAP of the Random Forest on the Boruta genes")
save(pd.DataFrame([{"set": "Boruta only", "n": len(bor_set - deg_set)}, {"set": "DEG only", "n": len(deg_set - bor_set)},
                   {"set": "Overlap", "n": len(overlap)}, {"set": "Boruta total", "n": len(bor_set)},
                   {"set": "DEG total", "n": len(deg_set)}]), "06_overlap_analysis.csv", "Venn counts")
save(memb, "06_gene_set_membership.csv", "Boruta / DE membership")
save(SG, "07_single_gene_auc.csv", "Out-of-fold single-gene AUC (10 x 5 CV, 63 people)")
save(ann, "13_gene_annotation_master.csv", "Per-gene summary for the Boruta genes")
save(pd.DataFrame(roc_rows), "15_roc_curves.csv", "Out-of-fold single-gene ROC coordinates")
for f in ("15_gene_symbol_map.csv", "01_preprocessing_summary.csv", "01_cohort_by_dataset.csv"):
    save(pd.read_csv(find_input(f)), f, "copied from the core notebook")
pd.DataFrame([{"n": i + 1, "file": k, "rows": v[0], "description": v[1]} for i, (k, v) in enumerate(W.items())]
             ).to_csv(OUT / "00_manifest.csv", index=False)
print(imp[["shap_rank", "symbol", "mean_abs_shap", "direction_sign", "boruta_fold_frequency", "core_model_rank"]].round(3).to_string(index=False))
log("done")